# 06 — Primary unsupervised model

This notebook fits the primary label-free detector on calibration telemetry.
It produces four complementary channels:

- rapid self evidence from empirical calibration tails;
- persistent drift evidence from a gap-reset CUSUM;
- peer deviation within a valid Telecom peer group;
- common-mode evidence at physical topology scopes.

PCA and Isolation Forest are fitted at the same time but remain challengers.
No evaluation truth is mounted or read here.


## 1. Setup


In [ ]:
from pathlib import Path
import os
import sys

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")


def find_repository(start=Path.cwd()):
    """Find the checked-out repository when Jupyter starts in any subfolder."""
    override = os.getenv("TELCO_PROJECT_ROOT")
    if override:
        candidates = [Path(override).expanduser().resolve()]
    else:
        start = start.resolve()
        candidates = [start, *start.parents]
        if "google.colab" in sys.modules:
            candidates += [
                Path("/content/drive/MyDrive/anomaly_detection"),
                Path("/content/drive/MyDrive/telco-anomaly-detection"),
            ]
    for candidate in candidates:
        if (candidate / "pyproject.toml").is_file() and (candidate / "configs").is_dir():
            return candidate.resolve()
    raise FileNotFoundError(
        "Open this notebook from the cloned repository, or set TELCO_PROJECT_ROOT."
    )


PROJECT_ROOT = find_repository()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import joblib
import shutil

import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

from telco_anomaly.detectors import (
    calibration_thresholds,
    fit_residual_bundle,
    fit_topology_reference,
    merge_score_files,
    score_residual_file,
    score_topology_file,
)
from telco_anomaly.io import (
    immutable_output_directory,
    load_config,
    read_json,
    resolve_data_root,
    write_json,
)

DATA_ROOT = resolve_data_root()
DATASET = os.getenv("TELCO_DATASET", "synthetic_pon")
default_core_runs = {
    "synthetic_pon": "synthetic_pon_core_v2",
    "ran_pm": "ran_pm_v1",
    "microsoft_optical": "microsoft_optical_v1",
}
if DATASET not in default_core_runs:
    raise ValueError(f"Model fitting is not configured for {DATASET!r}")
legacy_core = os.getenv("PON_CORE_RUN_ID") if DATASET == "synthetic_pon" else None
legacy_features = os.getenv("PON_FEATURE_RUN_ID") if DATASET == "synthetic_pon" else None
legacy_models = os.getenv("PON_MODEL_RUN_ID") if DATASET == "synthetic_pon" else None
CORE_RUN_ID = os.getenv("TELCO_CORE_RUN_ID", legacy_core or default_core_runs[DATASET])
FEATURE_RUN_ID = os.getenv(
    "TELCO_FEATURE_RUN_ID", legacy_features or f"{DATASET}_features_v2"
)
MODEL_RUN_ID = os.getenv(
    "TELCO_MODEL_RUN_ID", legacy_models or f"{DATASET}_models_v2"
)

RUN_ROOT = DATA_ROOT / "core" / DATASET / CORE_RUN_ID
CORE_ROOT = RUN_ROOT / "SPEC-CORE"
FEATURE_ROOT = DATA_ROOT / "features" / DATASET / FEATURE_RUN_ID
OUTPUT_ROOT = DATA_ROOT / "models" / DATASET / MODEL_RUN_ID

ALERT_POLICY = load_config("alert_policy", project_root=PROJECT_ROOT)
TOPOLOGY_POLICY = load_config("topology", project_root=PROJECT_ROOT)
catalogue = pd.read_parquet(CORE_ROOT / "metric_catalogue.parquet")
feature_manifest = read_json(FEATURE_ROOT / "feature_manifest.json")

if (RUN_ROOT / "SPEC-EVAL").exists():
    raise PermissionError("Model fitting must use the truth-unmounted run")

feature_paths = {
    name: FEATURE_ROOT / values["features"]
    for name, values in feature_manifest["partitions"].items()
}
display(pd.Series({
    "calibration_features": str(feature_paths["calibration"]),
    "dataset": DATASET,
    "development_features": str(feature_paths["development"]),
    "model_output": str(OUTPUT_ROOT),
}, name="value").to_frame())


## 2. Fit the frozen self-history reference


In [ ]:
MAX_TRAINING_ROWS = int(os.getenv("PON_MAX_TRAINING_ROWS", "150000"))
ISOLATION_TREES = int(os.getenv("PON_ISOLATION_TREES", "200"))
CADENCE_SECONDS = float(catalogue["expected_cadence_seconds"].dropna().mode().iloc[0])

if OUTPUT_ROOT.exists():
    model_manifest = read_json(OUTPUT_ROOT / "model_manifest.json")
    if model_manifest["core_fingerprint"] != feature_manifest["core_fingerprint"]:
        raise ValueError("Existing model belongs to a different canonical run")
    bundle = joblib.load(OUTPUT_ROOT / "residual_bundle.joblib")
    print("Using existing immutable model:", OUTPUT_ROOT)
else:
    bundle = fit_residual_bundle(
        feature_paths["calibration"],
        use_entity_reference=True,
        catalogue=catalogue,
        maximum_training_rows=MAX_TRAINING_ROWS,
        random_seed=42,
        isolation_trees=ISOLATION_TREES,
        fit_multivariate=True,
    )

display(pd.Series({
    "training_rows": bundle["training_rows"],
    "health_features": len(bundle["feature_columns"]),
    "entity_reference": bundle["use_entity_reference"],
    "empirical_tail_calibration": bool(bundle["tail_reference"]),
    "isolation_forest_fitted": bundle["isolation_forest"] is not None,
}, name="value").to_frame())


## 3. Score calibration and development without labels


In [ ]:
def score_self(partition, output):
    score_path = output / f"{partition}_self_scores.parquet"
    residual_path = output / f"{partition}_residuals.parquet"
    score_residual_file(
        bundle,
        feature_paths[partition],
        score_path,
        cadence_seconds=CADENCE_SECONDS,
        dispersion_window_seconds=feature_manifest["dispersion_window_seconds"],
        cusum_allowance=ALERT_POLICY["channels"]["persistent_drift"]["cusum_allowance"],
        residual_destination=residual_path,
    )
    return score_path, residual_path


def choose_peer_level(topology):
    policy = TOPOLOGY_POLICY["peer_policy"]
    minimum_group_size = int(policy["minimum_valid_peers"]) + 1
    minimum_coverage = float(policy["minimum_entity_coverage"])
    total_entities = topology["entity_id"].nunique()
    hierarchy = topology.groupby("group_type")["hierarchy_level"].median()
    configured = TOPOLOGY_POLICY["peer_policy"]["preferred_levels"]
    actual = topology["group_type"].drop_duplicates().tolist()
    candidates = [name for name in configured if name in actual]
    candidates += sorted(
        set(actual) - set(candidates),
        key=lambda name: hierarchy.get(name, -1), reverse=True,
    )
    for group_type in candidates:
        level = topology.loc[topology["group_type"].eq(group_type)]
        sizes = level.groupby("group_id")["entity_id"].nunique()
        eligible_groups = set(sizes.loc[sizes.ge(minimum_group_size)].index)
        covered = level.loc[level["group_id"].isin(eligible_groups), "entity_id"].nunique()
        if total_entities and covered / total_entities >= minimum_coverage:
            return group_type
    raise ValueError(
        "No topology level gives enough peers to the required share of entities"
    )


## 4. Add peer and common-mode topology evidence


In [ ]:
if not OUTPUT_ROOT.exists():
    with immutable_output_directory(OUTPUT_ROOT) as output:
        self_paths = {}
        residual_paths = {}
        for partition in ("calibration", "development"):
            self_paths[partition], residual_paths[partition] = score_self(partition, output)

        topology_path = CORE_ROOT / "topology_memberships.parquet"
        topology_enabled = topology_path.exists()
        topology_reason = "available"
        topology_reference = pd.DataFrame()
        peer_level = None

        if topology_enabled:
            topology = pd.read_parquet(topology_path)
            physical = topology.loc[topology["group_family"].eq("physical_topology")]
            try:
                peer_level = choose_peer_level(physical)
                group_levels = physical.sort_values(
                    "hierarchy_level", ascending=False
                )["group_type"].drop_duplicates().tolist()
                topology_reference = fit_topology_reference(
                    residual_paths["calibration"],
                    topology,
                    bundle["feature_columns"],
                    peer_group_type=peer_level,
                    group_types=group_levels,
                    min_peers=TOPOLOGY_POLICY["peer_policy"]["minimum_valid_peers"],
                    min_group_entities=TOPOLOGY_POLICY["group_policy"]["minimum_entities"],
                    min_group_fraction=TOPOLOGY_POLICY["group_policy"]["minimum_available_fraction"],
                )
            except ValueError as error:
                topology_enabled = False
                topology_reason = str(error)

        combined_paths = {}
        for partition in ("calibration", "development"):
            combined = output / f"{partition}_scores.parquet"
            if topology_enabled:
                topology_scores = output / f"{partition}_topology_scores.parquet"
                score_topology_file(
                    residual_paths[partition], topology, topology_reference,
                    topology_scores,
                    peer_group_type=peer_level,
                    group_types=group_levels,
                    min_peers=TOPOLOGY_POLICY["peer_policy"]["minimum_valid_peers"],
                    min_group_entities=TOPOLOGY_POLICY["group_policy"]["minimum_entities"],
                    min_group_fraction=TOPOLOGY_POLICY["group_policy"]["minimum_available_fraction"],
                )
                merge_score_files(self_paths[partition], topology_scores, combined)
            else:
                shutil.copy2(self_paths[partition], combined)
            combined_paths[partition] = combined

        score_columns = set(pq.ParquetFile(combined_paths["calibration"]).schema_arrow.names)
        channels = [
            name for name in (
                "rapid_residual", "drift_cusum", "peer_deviation",
                "group_common_mode", "dispersion_change", "pca_spe",
                "isolation_forest",
            )
            if name in score_columns
        ]
        thresholds = calibration_thresholds(
            combined_paths["calibration"],
            ALERT_POLICY["thresholds"]["candidate_quantiles"],
            block_column="entity_id",
            block_duration_seconds=ALERT_POLICY["thresholds"]["block_seconds"],
            minimum_block_rows=max(4, round(0.5 * 86_400 / CADENCE_SECONDS)),
            model_ids=channels,
        )

        joblib.dump(bundle, output / "residual_bundle.joblib")
        thresholds.to_parquet(output / "calibration_thresholds.parquet", index=False)
        if len(topology_reference):
            topology_reference.to_parquet(output / "topology_reference.parquet", index=False)
        model_manifest = {
            "dataset": DATASET,
            "core_fingerprint": feature_manifest["core_fingerprint"],
            "calibration_only_fit": True,
            "truth_files_read": [],
            "channels": channels,
            "primary_channels": [
                "rapid_residual", "drift_cusum", "peer_deviation", "group_common_mode"
            ],
            "challenger_channels": ["dispersion_change", "pca_spe", "isolation_forest"],
            "score_files": {
                name: f"{name}_scores.parquet" for name in combined_paths
            },
            "feature_columns": bundle["feature_columns"],
            "topology_enabled": topology_enabled,
            "topology_status": topology_reason,
            "peer_level": peer_level,
            "cadence_seconds": CADENCE_SECONDS,
            "dispersion_window_seconds": feature_manifest["dispersion_window_seconds"],
            "seasonal_periods": feature_manifest["seasonal_periods"],
            "threshold_method": "empirical quantiles of calibration daily block maxima",
        }
        write_json(output / "model_manifest.json", model_manifest)

thresholds = pd.read_parquet(OUTPUT_ROOT / "calibration_thresholds.parquet")
display(thresholds)


## 5. Acceptance


In [ ]:
model_manifest = read_json(OUTPUT_ROOT / "model_manifest.json")
assert model_manifest["calibration_only_fit"] is True
assert model_manifest["truth_files_read"] == []
assert {"rapid_residual", "drift_cusum"} <= set(model_manifest["channels"])

display(pd.Series({
    "primary_channels_available": [
        name for name in model_manifest["primary_channels"]
        if name in model_manifest["channels"]
    ],
    "topology_status": model_manifest["topology_status"],
    "challengers_available": [
        name for name in model_manifest["challenger_channels"]
        if name in model_manifest["channels"]
    ],
}, name="result").to_frame())
print("PASS — calibration-frozen scores and thresholds are ready")
print("Next: 07_CHALLENGER_MODELS.ipynb")
